In [9]:
# STEP 1.1: Download the dataset
from datasets import load_dataset

ds = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    split="train",
)

print("Number of examples:", ds.num_rows)
print("Columns:", ds.column_names)
print("\n----- Example #1 -----")
for key, value in ds[0].items():
    print(f"{key}: {value}")

Number of examples: 26872
Columns: ['flags', 'instruction', 'category', 'intent', 'response']

----- Example #1 -----
flags: B
instruction: question about cancelling order {{Order Number}}
category: ORDER
intent: cancel_order
response: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.


In [10]:
# STEP 1.2: Explore the data
import pandas as pd
pd.set_option("display.max_rows", 50)

df = ds.to_pandas()   # turn the dataset into a table (like Excel)

# --- 1) Topics ---
print("Number of categories:", df["category"].nunique())
print("Number of intents   :", df["intent"].nunique())
print("\n--- Examples per CATEGORY ---")
print(df["category"].value_counts())
print("\n--- Examples per INTENT ---")
print(df["intent"].value_counts())

# --- 2) Lengths (in words) ---
df["q_words"] = df["instruction"].str.split().str.len()
df["a_words"] = df["response"].str.split().str.len()
print("\n--- Question length (words) ---")
print(df["q_words"].describe().round(1))
print("\n--- Answer length (words) ---")
print(df["a_words"].describe().round(1))

# --- 3) A few real examples from different intents ---
for intent in ["cancel_order", "get_refund", "track_order"]:
    row = df[df["intent"] == intent].sample(1, random_state=42).iloc[0]
    print(f"\n========== {intent} ==========")
    print("CUSTOMER:", row["instruction"])
    print("AGENT   :", row["response"][:600], "...")

print("######################################")
print("Checking if all intents have equal proportions")
counts = df["intent"].value_counts()
print(counts)
print("Number of intents:", counts.size)
print("Smallest:", counts.min(), "| Largest:", counts.max())

Number of categories: 11
Number of intents   : 27

--- Examples per CATEGORY ---
category
ACCOUNT         5986
ORDER           3988
REFUND          2992
CONTACT         1999
INVOICE         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64

--- Examples per INTENT ---
intent
contact_customer_service    1000
complaint                   1000
check_invoice               1000
switch_account              1000
edit_account                1000
contact_human_agent          999
check_payment_methods        999
delivery_period              999
newsletter_subscription      999
get_invoice                  999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998
change_order                 997
set_up_shipping_address      997
check_refund_policy          997
create_account         

In [11]:
# STEP 1.3: Set aside the exam (test set)
from sklearn.model_selection import train_test_split

# Take 200 examples out, spread evenly across all intents
pool_df, test_df = train_test_split(
    df,
    test_size=200,
    stratify=df["intent"],   # keep the same mix of intents
    random_state=42,         # same random pick every time
)

# Safety: remove any training-pile rows whose question is EXACTLY the same as an exam question
exam_questions = set(test_df["instruction"].str.strip().str.lower())
before = len(pool_df)
pool_df = pool_df[~pool_df["instruction"].str.strip().str.lower().isin(exam_questions)]
removed = before - len(pool_df)

print("Exam (test) size     :", len(test_df))
print("Training pile size   :", len(pool_df))
print("Duplicates removed   :", removed)
print("Intents in exam      :", test_df["intent"].nunique())
print("Overlap of questions :", len(exam_questions & set(pool_df["instruction"].str.strip().str.lower())))
print("\n--- Exam questions per intent ---")
print(test_df["intent"].value_counts().to_string())

Exam (test) size     : 200
Training pile size   : 26593
Duplicates removed   : 79
Intents in exam      : 27
Overlap of questions : 0

--- Exam questions per intent ---
intent
newsletter_subscription     8
get_invoice                 8
registration_problems       8
delivery_period             8
contact_customer_service    8
check_invoice               8
payment_issue               8
edit_account                8
switch_account              8
complaint                   8
contact_human_agent         8
track_order                 7
check_payment_methods       7
set_up_shipping_address     7
delivery_options            7
delete_account              7
place_order                 7
check_refund_policy         7
cancel_order                7
get_refund                  7
change_order                7
check_cancellation_fee      7
track_refund                7
create_account              7
change_shipping_address     7
recover_password            7
review                      7


In [12]:
# STEP 1.4: Pick balanced training + validation sets
TRAIN_PER_INTENT = 150   # 150 x 27 intents = about 4,050 training examples
VAL_PER_INTENT   = 8     # 8 x 27 intents   = about 216 validation examples

# Remove exact duplicate questions, then shuffle randomly (same way every time)
shuffled = pool_df.drop_duplicates("instruction").sample(frac=1, random_state=42)

# Take the first 150 of each intent for training
train_df = shuffled.groupby("intent").head(TRAIN_PER_INTENT)

# From what's left, take 8 of each intent for validation
rest   = shuffled.drop(train_df.index)
val_df = rest.groupby("intent").head(VAL_PER_INTENT)

# Shuffle the training set so topics are mixed, not grouped together
train_df = train_df.sample(frac=1, random_state=42)

# --- Checks ---
q = lambda d: set(d["instruction"].str.strip().str.lower())
print("Training size   :", len(train_df))
print("Validation size :", len(val_df))
print("Test (exam) size:", len(test_df))
print("Intents in train:", train_df["intent"].nunique(),
      "| min per intent:", train_df["intent"].value_counts().min(),
      "| max per intent:", train_df["intent"].value_counts().max())
print("Intents in val  :", val_df["intent"].nunique())
print("Overlap train-val :", len(q(train_df) & q(val_df)))
print("Overlap train-test:", len(q(train_df) & q(test_df)))
print("Overlap val-test  :", len(q(val_df) & q(test_df)))
print("\nFirst 10 training intents (should look mixed):")
print(train_df["intent"].head(10).tolist())

Training size   : 4050
Validation size : 216
Test (exam) size: 200
Intents in train: 27 | min per intent: 150 | max per intent: 150
Intents in val  : 27
Overlap train-val : 0
Overlap train-test: 0
Overlap val-test  : 0

First 10 training intents (should look mixed):
['check_refund_policy', 'contact_customer_service', 'complaint', 'delivery_options', 'newsletter_subscription', 'review', 'create_account', 'change_order', 'newsletter_subscription', 'edit_account']


In [14]:
# ===== STEP 1.5: Convert to Mistral's conversation format =====
from datasets import Dataset
from transformers import AutoTokenizer

# 1) Turn each table row into a conversation (user -> assistant)
def to_chat(d):
    return Dataset.from_list([
        {
            "messages": [
                {"role": "user",      "content": r.instruction.strip()},
                {"role": "assistant", "content": r.response.strip()},
            ],
            "intent": r.intent,       # kept for analysis later, not trained on
            "category": r.category,
        }
        for r in d.itertuples()
    ])

train_ds = to_chat(train_df)
val_ds   = to_chat(val_df)
test_ds  = to_chat(test_df)
print("Sizes -> train:", len(train_ds), "| val:", len(val_ds), "| test:", len(test_ds))

# 2) Load ONLY Mistral's tokenizer (its "reader/formatter"), not the whole model (small download)
tok = AutoTokenizer.from_pretrained("unsloth/mistral-7b-instruct-v0.3")

# 3) See exactly how Mistral will read one training example
example = train_ds[0]
print("\n----- As a conversation -----")
print(example["messages"])
print("\n----- As Mistral sees it -----")
print(tok.apply_chat_template(example["messages"], tokenize=False))

Sizes -> train: 4050 | val: 216 | test: 200

----- As a conversation -----
[{'content': 'i dont know how to see ur money back policy', 'role': 'user'}, {'content': 'I\'ve understood, you need assistance in finding our money back policy. To help you with that, you can visit our website and navigate to the "Refund Policy" page. This page will provide you with all the information you need regarding our refund process, including the situations in which you can request a refund and the steps to follow.\n\nAdditionally, if you have any specific questions or concerns about our refund policy, feel free to let me know. I\'m here to provide you with any additional information or clarification you may need. Your satisfaction is our top priority.', 'role': 'assistant'}]

----- As Mistral sees it -----
<s>[INST] i dont know how to see ur money back policy[/INST] I've understood, you need assistance in finding our money back policy. To help you with that, you can visit our website and navigate to th

In [15]:
# STEP 1.6: Check token lengths
import numpy as np

# Count tokens for each example (in Mistral's format, exactly as the model sees it)
def count_tokens(ex):
    text = tok.apply_chat_template(ex["messages"], tokenize=False)
    return {"n_tokens": len(tok(text, add_special_tokens=False)["input_ids"])}

train_ds = train_ds.map(count_tokens)
val_ds   = val_ds.map(count_tokens)

lengths = np.array(train_ds["n_tokens"])
print("--- Training example length (tokens) ---")
print("Shortest      :", lengths.min())
print("Average       :", int(lengths.mean()))
print("Half are under:", int(np.percentile(lengths, 50)))
print("95% are under :", int(np.percentile(lengths, 95)))
print("99% are under :", int(np.percentile(lengths, 99)))
print("Longest       :", lengths.max())

print("\n--- How many would be TOO LONG for each limit? ---")
for limit in [256, 512, 768, 1024]:
    too_long = (lengths > limit).sum()
    print(f"Limit {limit:>4}: {too_long:>4} too long ({too_long/len(lengths)*100:.1f}%)")

# Our choice
MAX_LEN = 1024
before_t, before_v = len(train_ds), len(val_ds)
train_ds = train_ds.filter(lambda ex: ex["n_tokens"] <= MAX_LEN)
val_ds   = val_ds.filter(lambda ex: ex["n_tokens"] <= MAX_LEN)
print(f"\nMAX_LEN = {MAX_LEN}")
print("Train kept:", len(train_ds), "of", before_t)
print("Val kept  :", len(val_ds), "of", before_v)

Map:   0%|          | 0/4050 [00:00<?, ? examples/s]

Map:   0%|          | 0/216 [00:00<?, ? examples/s]

--- Training example length (tokens) ---
Shortest      : 27
Average       : 150
Half are under: 127
95% are under : 302
99% are under : 416
Longest       : 517

--- How many would be TOO LONG for each limit? ---
Limit  256:  377 too long (9.3%)
Limit  512:    2 too long (0.0%)
Limit  768:    0 too long (0.0%)
Limit 1024:    0 too long (0.0%)


Filter:   0%|          | 0/4050 [00:00<?, ? examples/s]

Filter:   0%|          | 0/216 [00:00<?, ? examples/s]


MAX_LEN = 1024
Train kept: 4050 of 4050
Val kept  : 216 of 216


In [16]:
# ===== STEP 1.7: Save to Google Drive =====
import os, json
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/mistral-finetune/data"
os.makedirs(DATA_DIR, exist_ok=True)

# 1) Save the three sets as .jsonl files
train_ds.to_json(f"{DATA_DIR}/train.jsonl")
val_ds.to_json(f"{DATA_DIR}/val.jsonl")
test_ds.to_json(f"{DATA_DIR}/test.jsonl")

# 2) Save a small notes file with our choices
info = {
    "source_dataset": "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    "model_tokenizer": "unsloth/mistral-7b-instruct-v0.3",
    "train_size": len(train_ds),
    "val_size": len(val_ds),
    "test_size": len(test_ds),
    "train_per_intent": TRAIN_PER_INTENT,
    "val_per_intent": VAL_PER_INTENT,
    "max_len_tokens": MAX_LEN,
    "random_seed": 42,
}
with open(f"{DATA_DIR}/data_info.json", "w") as f:
    json.dump(info, f, indent=2)

print("Saved files:")
for name in sorted(os.listdir(DATA_DIR)):
    size_kb = os.path.getsize(f"{DATA_DIR}/{name}") / 1024
    print(f"  {name:<16} {size_kb:8.1f} KB")

Mounted at /content/drive


Creating json from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved files:
  data_info.json        0.3 KB
  test.jsonl          156.6 KB
  train.jsonl        3257.7 KB
  val.jsonl           175.7 KB
